In [13]:
# ============================================
# 1. IMPORT LIBRARIES
# ============================================

import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.sentiment import SentimentIntensityAnalyzer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

nltk.download('stopwords')
nltk.download('vader_lexicon')

stop_words = set(stopwords.words("english"))


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\91908\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\91908\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [16]:


# ============================================
# 2 LOAD FAKE REVIEW DATASET (TRAINING DATA)
# ============================================

df = pd.read_csv("fake reviews dataset.csv")

print("Original dataset shape:", df.shape)

review_col = "review"
label_col = "label"


# ============================================
# 3 REMOVE DUPLICATES
# ============================================

df = df.drop_duplicates(subset=[review_col])


# ============================================
# 4 CLEAN TEXT FUNCTION
# ============================================

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"[^\w\s]", "", text)

    words = text.split()
    words = [w for w in words if w not in stop_words]

    return " ".join(words)


df["clean_review"] = df[review_col].apply(clean_text)


# ============================================
# 5 FILTER EDUCATION DOMAIN REVIEWS
# ============================================

education_keywords = [
    "college","university","campus","faculty","professor",
    "teacher","course","classes","students","education",
    "degree","lecture","department","placement",
    "internship","hostel","library","lab","exam","semester"
]

def is_education_related(text):
    text = str(text).lower()
    return any(word in text for word in education_keywords)

df_filtered = df[df["clean_review"].apply(is_education_related)]

print("After education filtering:", df_filtered.shape)


# ============================================
# 6 FIX LABELS (OR / CG → Genuine / Fake)
# ============================================

print("Original labels:", df_filtered[label_col].unique())

df_filtered[label_col] = df_filtered[label_col].map({
    "OR": "Genuine",
    "CG": "Fake"
})

print("Mapped labels:")
print(df_filtered[label_col].value_counts())


Original dataset shape: (40432, 4)
After education filtering: (2072, 5)
Original labels: ['CG' 'OR']
Mapped labels:
label
Genuine    1476
Fake        596
Name: count, dtype: int64


C:\Users\91908\AppData\Local\Temp\ipykernel_63484\617211869.py:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered[label_col] = df_filtered[label_col].map({


In [21]:
# ============================================
# 7 PREPARE TRAINING DATA
# ============================================

X = df_filtered["clean_review"]
y = df_filtered[label_col]


# ============================================
# 8 TF-IDF VECTORIZATION
# ============================================

vectorizer = TfidfVectorizer(max_features=5000)

X_vectorized = vectorizer.fit_transform(X)


# ============================================
# 9 TRAIN TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X_vectorized,
    y,
    test_size=0.2,
    random_state=42
)


# ============================================
# 10 TRAIN MACHINE LEARNING MODEL
# ============================================

model = LogisticRegression(max_iter=1000, class_weight="balanced")

model.fit(X_train, y_train)


# ============================================
# 11 EVALUATE MODEL
# ============================================

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))


# ============================================
# 12 LOAD SCRAPED EDUCATION REVIEWS
# ============================================

scraped = pd.read_csv("institutes.csv", encoding="latin-1")

print("Scraped dataset size:", scraped.shape)


# ============================================
# 13 CLEAN SCRAPED REVIEWS
# ============================================

scraped["clean_review"] = scraped["Reviews"].apply(clean_text)


# ============================================
# 14 TF-IDF TRANSFORM
# ============================================

scraped_features = vectorizer.transform(scraped["clean_review"])


# ============================================
# 15 PREDICT FAKE REVIEWS
# ============================================

scraped["Fake_Prediction"] = model.predict(scraped_features)

print("Prediction distribution:")
print(scraped["Fake_Prediction"].value_counts())

Accuracy: 0.908433734939759
              precision    recall  f1-score   support

        Fake       0.86      0.82      0.84       119
     Genuine       0.93      0.95      0.94       296

    accuracy                           0.91       415
   macro avg       0.89      0.88      0.89       415
weighted avg       0.91      0.91      0.91       415

Scraped dataset size: (356, 4)
Prediction distribution:
Fake_Prediction
Genuine    337
Fake        19
Name: count, dtype: int64


In [23]:
# ============================================
# 16 SENTIMENT ANALYSIS
# ============================================

sia = SentimentIntensityAnalyzer()

scraped["sentiment_score"] = scraped["Reviews"].apply(
    lambda x: sia.polarity_scores(str(x))["compound"]
)


# ============================================
# 17 SENTIMENT LABEL
# ============================================

def sentiment_label(score):

    if score > 0.05:
        return "Positive"

    elif score < -0.05:
        return "Negative"

    else:
        return "Neutral"


scraped["Sentiment"] = scraped["sentiment_score"].apply(sentiment_label)


# ============================================
# 18 CLEAN RATINGS
# ============================================

scraped["Ratings"] = scraped["Ratings"].astype(str)

scraped["Ratings"] = scraped["Ratings"].str.extract(r"(\d+\.\d+)")

scraped["Ratings"] = scraped["Ratings"].astype(float)


# ============================================
# 19 SAVE FINAL DATASET
# ============================================

scraped.to_csv("education_reviews_analysis_clean.csv", index=False)

print("Final dataset saved successfully")


# ============================================
# 20 PREVIEW FINAL DATA
# ============================================

print(scraped.head())

Final dataset saved successfully
                       University/College  Ratings  \
0                                    GECA      3.6   
1                              GIMS Noida      4.1   
2           Motilal Nehru Evening College      3.5   
3    Narayana Engineering College Nellore      4.8   
4  SRK Institute of Technology Vijayawada      5.0   

                                             Reviews       Source   \
0  there were 10 -15 faculty are in civil departm...  Collegedunia   
1  The course curriculum is updated every year as...  Collegedunia   
2  In regard to the relevancy of the course Im qu...  Collegedunia   
3  Students are eligible for the placements from ...  Collegedunia   
4  The faculty-to-student ratio is something like...  Collegedunia   

                                        clean_review Fake_Prediction  \
0  faculty civil department faculty permanent tem...         Genuine   
1  course curriculum updated every year per indus...         Genuine   
2  re

In [24]:
print(scraped["Fake_Prediction"].value_counts())

Fake_Prediction
Genuine    337
Fake        19
Name: count, dtype: int64
